# Companion — `Main_Demo.ipynb`

A reading companion for [`Main_Demo.ipynb`](./Main_Demo.ipynb). Open both side by side.

**How to use:** bullets are keyed `**[n]**` by cell index, and each carries a code anchor
(e.g. `**[37]** residual_stream_patching_hook`) because Colab hides cell numbers and disables
Ctrl+F — use the **sidebar search** on the anchor text to jump.

Setup cells get one line. Cells that teach something get three to five.

## Summary — what this notebook is actually doing

Main_Demo is a **tour of the instrument**, not a research result. It answers one question in five
movements: *how do I get at the numbers inside a transformer, and how do I change them?*

1. **Load and run** `[15–20]` — a model is a callable that returns `logits`, `loss`, `both`, or
   `None`. `None` is the one people forget: run the forward pass purely for its side effects when
   all you want is the cache.
2. **Read everything** `[21–27]` — `run_with_cache` hands back every intermediate activation.
   Then immediately: how *not* to, via `names_filter` and `stop_at_layer`, because caching a whole
   model is how you OOM.
3. **Write to it** `[28–39]` — a hook function is `(activation, hook) -> activation`. That single
   signature gets you ablation (zero a head) and activation patching (splice a clean activation
   into a corrupted run). The IOI patching heatmap at `[39]` is the notebook's real payoff: it
   localises a behaviour to specific layers and positions.
4. **Read without writing** `[40–48]` — same hook mechanism, return nothing. Used to hunt
   induction heads across all layers at once.
5. **Everything else** `[49–145]` — tokenization gotchas, `FactoredMatrix` for low-rank circuit
   analysis, and training checkpoints.

The conceptual spine is **`[29]`** (why intervention matters) and **`[42]`** (what an induction
head is). If you read only two markdown cells, read those.

**Watch for:** this notebook was ported to `TransformerBridge` (v3) but parts of its prose still
describe the old `HookedTransformer` API. Four places where the text and code disagree are flagged
below — `[56]`, `[65]`, `[133]`, `[143]`.

## Know by heart

Ten things. Everything else you can look up.

**1. Load a model**
```python
model = TransformerBridge.boot_transformers("openai-community/gpt2", device=device)
model.enable_compatibility_mode(disable_warnings=True)
```
The second line is not optional decoration — it gives you folded-LayerNorm numerics *and* the
classic `blocks.0.attn.hook_pattern` hook names that every tutorial assumes.

**2. Run it four ways**
```python
model(text, return_type="logits")  # [batch, pos, d_vocab]
model(text, return_type="loss")    # scalar cross-entropy
model(text, return_type="both")    # (logits, loss)
model(text, return_type=None)      # no output — you only want the hooks to fire
```

**3. Cache activations**
```python
logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
cache["blocks.0.attn.hook_pattern"]   # by full name
cache["pattern", 0, "attn"]           # same thing, shorthand
```

**4. Cache *less*** — the difference between fitting in memory and not:
```python
_, cache = model.run_with_cache(
    tokens, stop_at_layer=1, names_filter=["blocks.0.attn.hook_pattern"]
)
```

**5. The hook signature.** Every hook you will ever write is this shape:
```python
def my_hook(activation, hook: HookPoint):
    activation[:, :, head_index, :] = 0.0   # edit in place
    return activation                        # return to modify; return nothing to just observe
```

**6. Run with hooks** — temporary, removed after the call:
```python
model.run_with_hooks(
    tokens, return_type="loss",
    fwd_hooks=[(utils.get_act_name("v", layer), my_hook)],
)
```

**7. `utils.get_act_name`** builds hook names so you don't hand-format strings.
`get_act_name("v", 0)` → `"blocks.0.attn.hook_v"`. A name filter can also be a *function*:
`lambda name: name.endswith("hook_resid_post")`.

**8. `functools.partial` for extra hook args.** Hooks must be `(activation, hook)`, so bind
anything else first:
```python
partial(residual_stream_patching_hook, position=pos)
```

**9. `hook.name` and `hook.layer()`.** Inside a hook you can find out where you are — that's what
lets *one* function serve all 12 layers, writing into `store[hook.layer()]`.

**10. BOS is prepended by default.** `model("Hello World")` is 3 tokens, not 2. If you have an
off-by-one in a position index, this is why. `prepend_bos=False` turns it off.

**The five shapes**, for GPT-2 small: `n_layers=12`, `n_heads=12`, `d_model=768`, `d_head=64`,
`d_mlp=3072`. Attention activations are `[batch, pos, head_index, d_head]`; patterns are
`[batch, head, dest_pos, src_pos]` and lower-triangular because attention is causal.

## Setup `[0–12]`

- **[0]** Colab badge (points at your fork).
- **[1]** Title + the red "Runtime > Change Runtime Type > GPU" notice. Do it.
- **[2]** Reading tips — note "search the sidebar, not CTRL+F". Colab really does break Ctrl+F.
- **[3]** Header: "Setup (No need to read)".
- **[4]** Colab detection → `%pip install transformer_lens circuitsvis`. **This is where you get
  the "restart runtime" prompt** (Colab ships transformers 4.x; TL 3.8.1 needs ≥5.9.0). Accept,
  then re-run from here.
- **[5]** Plotly renderer. On Colab it picks `"colab"` — correct, leave it.
- **[6]** `cv.examples.hello("Neel")` — looks like filler, isn't. It's a smoke test for
  **circuitsvis**, which renders the interactive attention widgets. If nothing appears here, the
  visualisations at `[25]` and `[48]` will silently fail 40 cells later. Fix it now.
- **[7]** General imports. Three you should recognise on sight:
  `einops` (named reshaping — `rearrange`, `repeat`, `reduce`), `fancy_einsum` (einsum with
  readable names), `jaxtyping.Float` (shape annotations like `Float[Tensor, "batch pos d_model"]`
  — documentation, not enforcement).
- **[8]** The TransformerLens surface: `HookedTransformer` (legacy), `TransformerBridge` (v3, what
  this notebook loads), `HookPoint` (the core object), `FactoredMatrix` (low-rank products).
- **[9–10]** `torch.set_grad_enabled(False)`. Stated reason is GPU memory; the real point is that
  this notebook is *inference-time* analysis — forward passes, read activations, edit in flight.
  No training. You flip this back on for attribution patching later.
- **[11–12]** Plotting helpers `imshow` / `line` / `scatter`. Note `imshow` uses a red-blue
  diverging scale centred at zero — it assumes signed data, which is why patching results read
  well on it.

## Introduction `[13–14]`

- **[13–14]** Neel's framing of mech interp: we have programs that speak English and no idea how
  they work; the goal is to reverse-engineer learned algorithms from weights. Worth reading once
  for the motivation, then move on — no code.

## Loading and Running Models `[15–20]`

- **[15]** Section intro. 9,000+ models, one loader.
- **[16]** `device = utils.get_device()` — picks CUDA / MPS / CPU. On Colab with GPU enabled this
  should be `cuda`; print it if anything later is mysteriously slow.
- **[17]** **The most important cell in the notebook.**
  ```python
  model = TransformerBridge.boot_transformers("openai-community/gpt2", device=device)
  model.enable_compatibility_mode(disable_warnings=True)
  ```
  `boot_transformers` loads raw HuggingFace weights with architecture-native hook names.
  `enable_compatibility_mode()` then folds LayerNorm, centres the writing weights, and registers
  the classic HT hook aliases. Without that second line the numbers *and* the hook names differ
  from every tutorial you'll read. Don't replace this with `HookedTransformer.from_pretrained` —
  this is the modern equivalent.
- **[18–19]** `model(text, return_type="loss")` — the model is just callable. Cute detail: the
  text it computes loss on is the markdown of cell `[15]` itself.
- **[20]** The four `return_type` values. `None` is the one that matters later: run the forward
  pass for its hook side-effects only, skipping the logit computation entirely.

## Caching all Activations `[21–27]`

- **[21]** Section intro + the `remove_batch_dim` aside. A single-string input gives every
  activation a useless leading dim of 1; this strips it.
- **[22]** `logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)`. This is the
  single most-used call in mech interp. `cache` is an `ActivationCache` — dict-like, but with
  methods like `decompose_resid()` and `apply_ln_to_stack()` you'll meet in the IOI demo.
- **[23]** Explains the attention-pattern shape and circuitsvis.
- **[24]** `gpt2_cache["pattern", 0, "attn"]` — the tuple shorthand for
  `"blocks.0.attn.hook_pattern"`. Shape `[head_index, dest_pos, src_pos]` (batch dim already
  removed). Also `model.to_str_tokens` to get labels, since every attention weight connects a
  *pair* of tokens.
- **[25]** `cv.attention.attention_heads(...)` — interactive. Hover a head to isolate it. The grid
  is lower-triangular because a token can only attend backwards. Spend real time here; reading
  attention patterns is a skill you build by looking at hundreds of them.
- **[26–27]** **The memory lesson.** Same result, far cheaper:
  ```python
  _, cache = model.run_with_cache(
      tokens, stop_at_layer=1, names_filter=["blocks.0.attn.hook_pattern"]
  )
  ```
  `stop_at_layer` halts the forward pass early; `names_filter` stores only what you asked for. The
  `assert torch.allclose(...)` proves they agree. On bigger models this is not an optimisation,
  it's the difference between running and OOMing.

## Hooks: Intervening on Activations `[28–31]`

- **[28–29]** The conceptual core of the library. We have *full control*: every activation is
  wrapped in a `HookPoint`, and a hook function `(activation, hook) -> activation` can read,
  edit, or replace it. This is what makes causal claims possible — you're not correlating, you're
  intervening.
- **[30]** Sets up the ablation example and introduces `utils.get_act_name`.
- **[31]** **Ablation.** Zero out head 8's value vectors in layer 0, compare loss:
  ```python
  def head_ablation_hook(value, hook: HookPoint):
      value[:, :, head_index_to_ablate, :] = 0.0
      return value

  model.run_with_hooks(gpt2_tokens, return_type="loss",
                       fwd_hooks=[(utils.get_act_name("v", 0), head_ablation_hook)])
  ```
  Three things to absorb: `value` has shape `[batch, pos, head_index, d_head]` so head indexing is
  `[:, :, i, :]`; the edit is in-place (the `return` is convention); and `run_with_hooks` adds the
  hook *only for this call*, then removes it. Note the variable naming bug in the original —
  `layer_to_ablate`/`head_index_to_ablate` are 0 and 8, but the markdown at `[30]` says "head 7".
  Trust the code.

## Activation Patching on IOI `[32–39]`

The notebook's centrepiece. Read this section twice.

- **[32–33]** **Indirect Object Identification**: "After John and Mary went to the store, Mary gave
  a bottle of milk to" → " John". Activation patching (from the ROME paper) runs a *corrupted*
  prompt but splices in *clean* activations one at a time, asking: which activations, if restored,
  recover the correct behaviour?
- **[34–35]** The setup. Clean vs corrupted prompts differ by one token (Mary/John). The metric is
  a **logit difference**:
  ```python
  logits[0, -1, correct_index] - logits[0, -1, incorrect_index]
  ```
  Logit diff is the workhorse metric of mech interp — it cancels out everything the model does
  that isn't choosing between these two answers. Note `clean_cache` is saved here for splicing in.
- **[36–37]** **The patching loop.** The hook overwrites one position of the residual stream with
  its clean counterpart:
  ```python
  def residual_stream_patching_hook(resid_pre, hook: HookPoint, position: int):
      clean_resid_pre = clean_cache[hook.name]
      resid_pre[:, position, :] = clean_resid_pre[:, position, :]
      return resid_pre
  ```
  Two techniques here you'll reuse constantly: `hook.name` lets one function look up its own
  clean activation, and `functools.partial(..., position=pos)` binds the extra argument since
  hooks must be `(activation, hook)`. It sweeps every (layer, position) pair — that's
  `n_layers × n_positions` forward passes.
- **[38–39]** **The heatmap.** This is what a localised circuit looks like: the signal sits on the
  second "Mary" token through the early layers, then jumps to the final token at layers 7–8, where
  it's used to predict the answer. Those are the name-mover heads. Read the caveat in `[38]` —
  layers 7/8 not 8/9 because patching happens at `resid_pre`, the *start* of each layer.

## Hooks: Accessing Activations `[40–48]`

- **[40–41]** Same mechanism, read-only: return nothing and don't edit in place. Equivalent to
  `run_with_cache` + post-processing, but cheaper when you only need a scalar per head.
- **[42]** **What an induction head is** — the single most important concept in this notebook.
  A two-head circuit that continues repeated sequences: a **previous-token head** writes "the
  token before me was X" into the residual stream, and an **induction head** at a later layer uses
  that to attend from the current token to *the token after the previous occurrence of the current
  token*. That's how the model completes `... Michael Jordan ... Michael` → ` Jordan`.
- **[43–44]** **The behavioural test.** Generate 50 random tokens, repeat them twice, plot loss by
  position. Loss is terrible in the first half (unpredictable random tokens) and drops off a cliff
  at the halfway point — the model has no knowledge here except "I've seen this sequence before",
  so the drop *is* induction. `einops.repeat(x, "batch seq_len -> batch (2 seq_len)")` builds it.
- **[45–46]** **Scoring every head at once.**
  ```python
  induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
  induction_score = einops.reduce(induction_stripe, "batch head position -> head", "mean")
  induction_score_store[hook.layer(), :] = induction_score
  ```
  Why `offset = 1 - seq_len`? At destination position `seq_len + i` (the second copy of token `i`),
  an induction head attends to position `i + 1` — the token *after* the first occurrence. The gap
  is `(seq_len + i) - (i + 1) = seq_len - 1`, so that diagonal is the induction signal. Two
  reusable tricks: `hook.layer()` lets one function serve every layer, and the hook writes into a
  pre-allocated tensor rather than returning anything.
- **[47–48]** **Confirmation.** L5H5 scores highest; feeding it a short repeated sequence and
  visualising the pattern shows the "induction stripe" — a diagonal line offset from the main
  diagonal. You've now gone from behaviour → score → mechanism, which is the whole method in
  miniature.

## Available Models `[49–55]`

- **[49–50]** One loader for 9,000+ models; swap the name and re-run the analysis.
- **[51–52]** Re-runs the entire induction-scoring pipeline on **distilgpt2** by changing one
  string. The point isn't distilgpt2 — it's that your analysis code is model-agnostic.
- **[53–54]** Model inventory. **Marked "last updated 2023" in the notebook itself** — treat as
  historical. The live list is the docs link in `[50]`. Still useful for Neel's
  interpretability-friendly toy models (`attn-only-1l` … `4l`, `solu-*`), which have ~200 training
  checkpoints each and are the right size for learning.
- **[55]** Neel's resource list — the getting-started guide, glossary, and open-problems doc.
  These are the same links in `docs/source/content/getting_started_mech_interp.md`.

## HookedTransformer architecture `[56–63]`

> ⚠️ **Prose/code mismatch.** This section is written about `HookedTransformer`, but the model you
> loaded at `[17]` is a `TransformerBridge` in compatibility mode. The *concepts* below are
> accurate and worth learning; just don't expect the class names to match what you're holding.

- **[56]** How HT reshapes GPT-2 for interpretability: `W_Q`/`W_K`/`W_V` are **three separate
  matrices** rather than one concatenated block, and head dimensions are **split out** rather than
  flattened. So activations are `[batch, pos, head_index, d_head]` and you can index a single head
  directly — which is exactly what made `[31]`'s ablation a one-liner.
- **[57–58]** The five hyperparameters. Memorise these for GPT-2 small: `n_layers=12`,
  `n_heads=12`, `d_model=768`, `d_head=64`, `d_mlp=3072`, `n_ctx=1024`. Note the convention:
  weights multiply on the **right** (`out = in @ W + b`), which is transposed from most textbooks.
- **[59–61]** `model.tl_named_parameters()` — prints every weight and its shape, split into
  per-block and embedding/unembedding. Run these and actually read the output; knowing that `W_O`
  is `[head_index, d_head, d_model]` saves you hours of shape errors.
- **[62–63]** **How to discover hook names.** Attach a printing hook to everything and run a short
  prompt:
  ```python
  def print_name_shape_hook_function(activation, hook):
      print(hook.name, activation.shape)
  model.run_with_hooks(test_prompt, return_type=None,
                       fwd_hooks=[(name_filter, print_name_shape_hook_function)])
  ```
  Note the filter is a *lambda over names*, not a list. Keep this snippet — it's the fastest way to
  answer "what is this hook called?" on an unfamiliar model.

## Folding LayerNorm `[64–70]`

- **[64–65]** LayerNorm is a nuisance for interpretability because it makes the model non-linear in
  a way that breaks clean circuit analysis. "Folding" absorbs the LN scale/bias into the following
  weight matrix, making the model mathematically equivalent but easier to reason about.
  > ⚠️ **The text says these are the `fold_ln` / `center_writing_weights` flags on
  > `HookedTransformer.from_pretrained`.** You never call that function — `enable_compatibility_mode()`
  > at `[17]` is what turned folding on for you. Don't try to pass these flags to
  > `boot_transformers`; they aren't there.
- **[66–68]** A genuinely delightful consequence: folding creates an **unembed bias** `b_U` that
  GPT-2 wasn't trained with, and it encodes **unigram frequency**. Sorting it shows common tokens
  (` the`, ` and`, punctuation) at the top and junk like ` RandomRedditor` at the bottom. The model
  learned "some tokens are just more likely" and stored it in the final LayerNorm bias.
- **[69–70]** **Why you should care.** That bias favours ` John` over ` Mary` by ~1.2 logits — a
  3.6× probability ratio, about a third of the entire IOI circuit's effect. A chunk of what looks
  like sophisticated circuitry at `[39]` is actually just "John is a more common token." This is
  the section's real lesson: **check your baseline before attributing behaviour to a circuit.**

## Dealing with tokens `[71–90]`

- **[71]** Section header for the "Features" half. Points at the Exploratory Analysis demo — your
  next notebook.
- **[72–73]** Tokenization overview. The method family lives on the model:
  `to_tokens`, `to_str_tokens`, `to_string`, `to_single_token`, `get_token_position`.
- **[74–75]** `model.to_str_tokens(text)` → list of substrings. The four observations here are
  worth internalising: tokens **include their leading space** (`|how|` ≠ `| how|`), common words
  are single tokens while rare ones fragment, and case matters.
- **[76–79]** `to_tokens` → integer tensor `[batch, pos]`. Passing a list of strings pads shorter
  ones; in GPT-2, BOS = EOS = PAD = `50256`, which is a classic source of confusion.
- **[80–81]** `to_single_token(" The")` → one integer, for indexing into logits. Note the
  logits shape `[batch, pos, d_vocab]` — there's a next-token prediction at *every* position, not
  just the last, because attention is causal.
- **[82–83]** `to_string` is the inverse. `.squeeze()` drops the dummy batch dim so you get a
  string rather than a list of one.
- **[84–87]** `get_token_position(token, text, mode="first"|"last")`. Returns a zero-indexed
  position — and it's shifted by one because of the BOS token, which is the segue to the next
  section.
- **[88–89]** Arithmetic tokenizes inconsistently: `"2342+2017=21445"` does not split into clean
  digits. Genuinely explains a lot about why LLMs are bad at maths.
- **[90]** **Practical advice worth following.** Choose prompts whose key words are single tokens,
  in the same position, same total length. Study IOI with ` Tim`, not ` Ne|el`. Multi-token words
  force the model to spend early layers assembling them, which muddies whatever you're studying.

## Gotcha: `prepend_bos` `[91–97]`

- **[91–92]** **If you get an off-by-one error, check `prepend_bos` first.** TransformerLens
  prepends `<|endoftext|>` to every input by default — including implicit calls like
  `model("Hello World")`.
- **[93]** The demonstration: `model("Hello World")` gives 3 positions, `prepend_bos=False` gives 2.
- **[94]** **Why it exists.** Attention patterns are probability distributions — they must sum to
  1, so a head with "nothing to do" still has to attend somewhere. Without BOS it dumps that
  attention on the first real token, corrupting it. BOS gives heads a place to rest. This matters
  little in training (long sequences) and enormously for short interpretability prompts.
- **[95]** Same prompt with and without BOS gives a materially different logit difference. Not a
  rounding error — a real change in behaviour.
- **[96–97]** A second gotcha stacked on the first: `"Claire"` at the start of a sentence (no
  leading space) is **two tokens**, while `" Claire"` is one. Note they pass `prepend_bos=False`
  here deliberately, because they're inspecting tokenization rather than running the model.

## Factored Matrix Class `[98–120]`

- **[98]** Transformers are full of low-rank products: `M = A @ B` where `M` is `[large, large]`
  but `A` is `[large, small]`. `FactoredMatrix` keeps the factors and computes norms, eigenvalues,
  and SVD **without ever materialising `M`**.
- **[99–101]** Basic construction. `FactoredMatrix(A, B)` matches `A @ B` on `.norm()`, and exposes
  `ldim` / `rdim` / `mdim` (left, right, and the small hidden dimension).
- **[102–103]** Rank-2 matrix in a 5×5 shape → only 2 non-zero eigenvalues and singular values. The
  factored class omits the structural zeros rather than returning them.
- **[104–107]** Multiplication picks the cheapest contraction order automatically. `.AB` collapses
  back to a dense tensor when you actually need one.
- **[108–109]** **The OV circuit.** `W_OV = W_V @ W_O` is the map determining *what information
  gets moved* from source position to destination position (as opposed to the QK circuit, which
  decides *where* to move it from). `model.OV` gives all heads at once, shape
  `[n_layers, n_heads, d_model, d_model]`, kept factored.
- **[110–112]** **The eigenvalue copying score:**
  ```python
  eigenvalues.sum(dim=-1).real / eigenvalues.abs().sum(dim=-1)
  ```
  Mechanically: the real part of the eigenvalue sum over the sum of magnitudes, so it lands in
  `[-1, 1]`. It's near **+1** when eigenvalues are positive and real — meaning the head maps
  directions to *themselves*, i.e. copies. Near 0 or negative means it's doing something else.
- **[113–114]** L11H11 scores high; its eigenvalues plotted in the complex plane cluster on the
  positive real axis, as the score implies.
- **[115–118]** **The showpiece.** The full circuit `W_E @ W_OV @ W_U` is `[50257, 50257]` — about
  10GB dense, per head, and there are 144 heads. `FactoredMatrix` computes copying scores for all
  of them in seconds because it never builds the thing. This cell is *why* the class exists.
- **[119–120]** Full-circuit and OV-only scores correlate but not perfectly, and Neel says outright
  he doesn't know what the outliers mean. Worth noticing: unexplained results are normal here.

## Generating Text `[121–123]`

- **[121–123]** `model.generate(prompt, max_new_tokens=50, temperature=0.7)`. Useful for sanity-
  checking that a model does what you think. The notebook says to prefer HuggingFace for serious
  generation — this exists for convenience, not quality.

## Hook Points `[124–132]`

- **[124–125]** The general claim: the hook system works on **any** PyTorch model, not just
  transformers. A `HookPoint` is an identity module; `HookedRootModule` supplies `reset_hooks`,
  `run_with_cache`, and `run_with_hooks`.
- **[126–128]** **A toy model worth reading carefully** — it's the whole library in 20 lines.
  Two layers computing `x → x² + 3 → x² − 4`:
  ```python
  class SquareThenAdd(nn.Module):
      def __init__(self, offset):
          self.offset = nn.Parameter(torch.tensor(offset))
          self.hook_square = HookPoint()      # identity, but nameable and hookable
      def forward(self, x):
          square = self.hook_square(x * x)     # wrap the value you want to expose
          return self.offset + square
  ```
  That's the entire trick: wrap any intermediate value in a `HookPoint()` and it becomes
  cacheable and editable by name.
- **[129–130]** `run_with_cache` on the toy model — the same API as GPT-2, on a model with two
  parameters. Good for building intuition without a GPU.
- **[131–132]** Intervening: force `layer2.hook_square` to zero and the output becomes `-4`,
  exactly as the arithmetic predicts. Returning a value from a hook **replaces** the activation.

## Loading Pre-Trained Checkpoints `[133–145]`

> ⚠️ **This is where `HookedTransformer` is still the working path.** Cell `[133]` says
> `boot_transformers` handles checkpoints for only a few families, and `[143]` accordingly calls
> `HookedTransformer.from_pretrained(model_name, checkpoint_index=index)`. So HT isn't purely
> legacy — for checkpoint loading it's what you use.

- **[133–135]** Models with ~200 saved training checkpoints let you study *how circuits form*.
  Prefer `checkpoint_index` (count 0…N) over `checkpoint_value` (raw token/step counts on an
  ad-hoc schedule).
- **[136]** `get_checkpoint_labels(model_name)` plots each model's checkpoint schedule — some are
  log-spaced (dense early in training), some linear.
- **[137–138]** **The induction head phase transition.** Induction heads don't fade in gradually;
  they appear over a narrow window, and in-context learning jumps when they do — visible as a bump
  in the loss curve. One of the most striking results in the field.
- **[139–140]** The test: repeated-random-token loss, via `evals.induction_loss`. Deliberately
  rough — 4 sequences, demonstration not replication.
- **[141–143]** ⚠️ **The heaviest cell in the notebook.** It loads five checkpoints of a 2L model
  in a loop and keeps every one in `checkpointed_models`. On a free-tier T4 this is the most likely
  place to OOM or hit a slow download. If it stalls, cut `checkpoint_indices` to `[10, 35, -1]` —
  you'll still see the transition.
- **[144–145]** The payoff plot: induction loss drops sharply between ~200M and ~500M tokens, on a
  log x-axis. Note Neel flagging that this is earlier than the paper and he isn't sure why —
  again, unexplained discrepancies are normal.

## Where to go next

1. **Re-run `[36–39]` with a different metric.** Swap logit difference for probability of the
   correct answer and see whether the heatmap changes. This is the fastest way to learn what logit
   diff is actually buying you.
2. **Patch something other than `resid_pre`.** Try `z` (per-head output) instead — you'll get
   head-level resolution rather than layer-level, which is what identifies specific name-mover heads.
3. **Run the induction scan `[45–46]` on a toy model** — `attn-only-2l` is small, fast, and was
   built for exactly this.
4. **Then move to [`Exploratory_Analysis_Demo.ipynb`](./Exploratory_Analysis_Demo.ipynb)** — the
   same IOI task, taken all the way to a circuit, with logit attribution and path patching.

**On stale prose:** the four ⚠️ flags above (`[56]`, `[65]`, `[133]`, `[143]`) are all the same
underlying issue — the notebook was ported to `TransformerBridge` but its explanatory text still
describes `HookedTransformer`. When code and prose disagree, trust the code.